In [ ]:
!pip install pyproj
!pip install -U numpy==1.26.4
!pip install transformers
!pip install -U accelerate
!pip install pandas
!pip install PyPDF2
!pip install NLTK
!pip install editdistance
!pip install paramiko
!pip install func_timeout

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [1]:
# import libraries
import urllib.request
import requests
import json
from pyproj import Transformer
import os
import time
import sys
from datetime import datetime
from xml.etree import ElementTree as ET
import logging
import nltk
import pandas as pd
import re

# import common functions
sys.path.insert(1, '../')
import common

# set name of module, to fetch info from config
module_name = "archis"





/home/alex/anaconda3/envs/agnes-bert-file-import/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False
cuda gpu number is 0


Some weights of the model checkpoint at alexbrandsen/ArcheoBERTje-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/home/alex/anaconda3/envs/agnes-bert-file-import/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more 

In [2]:
# get info from config file
config = common.get_config()

# # set up logging
# log_location = config['data_source'][module_name]['harvest_log_location']
# now = datetime.now()
# date = now.strftime("%Y-%m-%d")
# logfile = f"{log_location}harvest-log-{module_name}-{date}.log"
# logging.basicConfig(level=logging.DEBUG, filename=logfile, filemode="a+",
#                 format="%(asctime)-15s %(levelname)-8s %(message)s")

# log config info        
#pdf_folder = config['data_source'][module_name]['pdf_folder']
#pdf_folder = '/media/alex/Data/agnes_data/archis/Archis_rapporten_2023/docs'
pdf_folder = '/media/alex/Data/agnes_data/archis/Archis_rapporten_2023/docs2'
print(f'pdf_folder: {pdf_folder}')

json_folder = config['data_source'][module_name]['json_folder']
print(f'json_folder: {json_folder}')

html_folder = config['data_source'][module_name]['html_folder']
print(f'html_folder: {html_folder}')

language = config['data_source'][module_name]['language']
print(f'language: {language}')

bert_model = config['bert_models'][language]
print(f'bert_model: {bert_model}')


pdf_folder: /media/alex/Data/agnes_data/archis/Archis_rapporten_2023/docs2
json_folder: /media/alex/Data/agnes_data/archis/json/
html_folder: /media/alex/Data/agnes_data/archis/html/
language: dutch
bert_model: /media/alex/Data/agnes_models/ArcheoBERTje-NER


In [3]:
archis_zaakdocumenten_location = '/media/alex/Data/agnes_data/archis/Archis_rapporten_2023/archis3_alle_zaakdocumenten.csv'
archis_zaken_location = '/media/alex/Data/agnes_data/archis/Archis_rapporten_2023/archis3_alle_zaken_select.csv'

archis_zaakdocumenten = pd.read_csv(archis_zaakdocumenten_location, sep=";")
archis_zaken = pd.read_csv(archis_zaken_location, sep=";")
print(archis_zaakdocumenten)

        document_id  zaakidentificatie     zaak_id  \
0         2006374.0         2325853100   2036202.0   
1         2006024.0         2326225100   2036243.0   
2         2006299.0         2329571100   2036616.0   
3         2006307.0         2329677100   2036627.0   
4         2000477.0         2329960100   2036659.0   
...             ...                ...         ...   
140563          NaN         5452842100  10110928.0   
140564          NaN         5453158100  10110963.0   
140565          NaN         5453400100  10110991.0   
140566   11941696.0         5461396100  10111882.0   
140567          NaN         5461396100  10111882.0   

                               identificatie  archis2_rapportmeldingsnr  \
0       9a3acbb3-b110-4515-9f27-bf87c229f01b                    23792.0   
1       0d24491e-00b7-4106-9a69-e6051a3b5dfc                    22009.0   
2       d27485e2-56ce-451a-8b65-a582afdc1a12                    31639.0   
3       c4a4b230-b70f-4e41-af98-a308ac9375f6       

In [9]:


for directory, subdirectories, files in os.walk(pdf_folder):
    for file in files:
        
        file_location = os.path.join(directory, file)

        # if not done yet
        file_name = common.cleanFileName(file)
        archis_zaakidentificatie = int(file.split('_')[0][1:]+'100')
        doc_id = f"{archis_zaakidentificatie}_{file_name.replace('.pdf','')}"
        json_output_folder = f"{json_folder}/{doc_id}"
        if os.path.exists(json_output_folder):
            print('Output folder for '+file+' already exists, skipping')
            continue #skip this file if it already exists
            
        print(f"indexing: {file}")
 
        archis_zaakidentificatie = int(file.split('_')[0][1:]+'100')
        #print(archis_zaakidentificatie)
        
        zaakdocument = archis_zaakdocumenten.loc[archis_zaakdocumenten['zaakidentificatie'] == archis_zaakidentificatie]
        zaak = archis_zaken.loc[archis_zaken['zaakidentificatie'] == archis_zaakidentificatie]
        
        if len(zaakdocument) == 0: # no result in db, log and skip
            print(f"no entry in db for {archis_zaakidentificatie}, skipping")
            continue
            
        if len(zaakdocument) > 1: # multiple rows with same zaakidentificatie, get one with document_id
            zaakdocument = zaakdocument.loc[zaakdocument['document_id'].notnull()]

        print(zaakdocument)
        
        file_name = common.cleanFileName(file)

        output_document = {}
        output_document['source'] = 'archis'
        output_document['file_name'] = file_name
        output_document['file_type'] = 'report'
        output_document['title'] = zaakdocument['titel'].values[0]
        if pd.notna(zaakdocument['auteur'].values[0]): # check if auteur is empty
            creators = re.split(',|&| en |/|;', zaakdocument['auteur'].values[0])    # split on muliple characters, because messy data         
            output_document['creators'] = creators
        output_document['description'] = '' # no descriptions in this data source
        output_document['publisher'] = zaak['uitvoerder'].values[0]
        if pd.notna(zaakdocument['jaar'].values[0]): # check if jaar is empty
            output_document['createdAt'] = int(zaakdocument['jaar'].values[0])
        output_document['identifiers'] = {
            'uri': zaakdocument['link'].values[0],
            'archis_zaakidentificatie': int(zaakdocument['zaakidentificatie'].values[0]),
            'archis_zaak_id': int(zaakdocument['zaak_id'].values[0]),
            'archis_identificatie': str(zaakdocument['identificatie'].values[0])
        }
        output_document['language'] = 'Dutch'
        output_document['html_folder_name'] = f"{archis_zaakidentificatie}_{file_name.replace('.pdf','')}"

        #coordinates
        if str(zaak['x_coordinaat'].values[0]) != 'nan' and str(zaak['x_coordinaat'].values[0]) != 'nan':
            coordX = int(zaak['x_coordinaat'].values[0])
            coordY = int(zaak['y_coordinaat'].values[0])
            lat, lon = common.rd2wgs(coordX,coordY)
            output_document['coordX'] = coordX
            output_document['coordY'] = coordY
            output_document['location'] = {'lat':lat,'lon':lon}

        # set document identifier
        doc_id = f"{archis_zaakidentificatie}_{file_name.replace('.pdf','')}"
        print(f"doc_id: {doc_id}")

        # save document.json 
        json_output_folder = f"{json_folder}/{doc_id}"
        common.savejson(output_document, f"{json_output_folder}/document.json")

        print(f"saved doc json")

        # process pdf, store page.json files with entities 
        common.run_ner_on_pdf(
            file_location, 
            json_output_folder, 
            bert_model, 
            language
        )

        print(f"ran NER, saved page json")

        # process pdf, save html files
        html_output_folder = f"{html_folder}/{doc_id}"
        common.pdf2html(file_location, html_output_folder)

        print(f"generated and saved html")



print('done!')


Output folder for Z5322520_08080701-afm-1696579604926-Archeologisch onderzoek Van Grevenbro.pdf already exists, skipping
indexing: Z5454721_34366966-afm-1695206098061-AD_BOPvEDurgerdammerdijk46_10-.pdf
no entry in db for 5454721100, skipping
Output folder for Z5403967_34137810-afm-1696235851208-RAAPrap_6468_RENB2_20230511.pdf already exists, skipping
Output folder for Z5025761_30129769-afm-1694699541060-SWNL0275803 Archeologisch bureauonder.pdf already exists, skipping
indexing: Z5310808_17138633-afm-1694778559774-230307_A22038_BU_Definitief.pdf
no entry in db for 5310808100, skipping
Output folder for Z5443843_34137810-afm-1695284676702-RAAPrap_6585_RUKER_20230814.pdf already exists, skipping
Output folder for Z5314453_13038286-afm-1694604634289-Eindrapportage archeologisch verkenne.pdf already exists, skipping
Output folder for Z5448306_64969533-afm-1694426403152-RER 122 - Bureauonderzoek archeologie.pdf already exists, skipping
Output folder for Z5450947_14117581-afm-1695207110151-A

IndexError: index 0 is out of bounds for axis 0 with size 0

In [3]:
# upload json and html to webserver
common.upload2webserver(json_folder, html_folder, module_name, config['webserver']['json_folder'], config['webserver']['html_folder'])

print(f"uploaded json/html to webserver")

# remotely start indexing script on webserver
common.start_index(module_name)

print(f"indexing on webserver started")

print(f"done!")

uploaded json/html to webserver
indexing on webserver started
done!
